<a href="https://colab.research.google.com/github/QAcdc5NA/kaua-simplicio-atividades-qa/blob/main/QA_SWAPI_atividade_Kaua_Santos_Simplicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade de QA com APIs — SWAPI (versão do aluno)

**Tempo estimado:** 30–40 minutos  
**Ferramenta:** Google Colab  
**Objetivo:** explorar uma API pública, analisar respostas JSON e criar testes simples com Python.

> Este notebook tenta consultar a SWAPI online. Se a internet falhar, ele usa um modo de fallback local para você conseguir concluir a atividade.

## Instruções

Nesta atividade, você vai:

1. consultar endpoints da SWAPI
2. observar o `status code`
3. ler o JSON retornado
4. pensar como QA
5. automatizar pelo menos 3 validações com `assert`

In [23]:
import requests
import json

FIXTURES = {
    "https://swapi.dev/api/people/10/": {
        "status_code": 200,
        "json": {
            "name": "Obi-Wan Kenobi",
            "height": "182",
            "mass": "77",
            "hair_color": "auburn, white",
            "skin_color": "fair",
            "eye_color": "blue-gray",
            "birth_year": "57BBY",
            "gender": "male",
            "homeworld": "https://swapi.dev/api/planets/20/",
            "films": [
                "https://swapi.dev/api/films/1/",
                "https://swapi.dev/api/films/2/",
                "https://swapi.dev/api/films/3/",
                "https://swapi.dev/api/films/4/",
                "https://swapi.dev/api/films/5/",
                "https://swapi.dev/api/films/6/"
            ],
            "species": [],
            "vehicles": [
                "https://swapi.dev/api/vehicles/38/"
            ],
            "starships": [
                "https://swapi.dev/api/starships/48/",
                "https://swapi.dev/api/starships/59/",
                "https://swapi.dev/api/starships/64/",
                "https://swapi.dev/api/starships/65/",
                "https://swapi.dev/api/starships/74/"
            ],
            "created": "2014-12-10T16:16:29.192000Z",
            "edited": "2014-12-20T21:17:50.325000Z",
            "url": "https://swapi.dev/api/people/10/"
        }
    },
    "https://swapi.dev/api/people/13/": {
        "status_code": 200,
        "json": {
            "name": "Chewbacca",
            "height": "228",
            "mass": "112",
            "hair_color": "brown",
            "skin_color": "unknown",
            "eye_color": "blue",
            "birth_year": "200BBY",
            "gender": "male",
            "homeworld": "https://swapi.dev/api/planets/14/",
            "films": [
                "https://swapi.dev/api/films/1/",
                "https://swapi.dev/api/films/2/",
                "https://swapi.dev/api/films/3/",
                "https://swapi.dev/api/films/6/"
            ],
            "species": [
                "https://swapi.dev/api/species/3/"
            ],
            "vehicles": [
                "https://swapi.dev/api/vehicles/19/"
            ],
            "starships": [
                "https://swapi.dev/api/starships/10/",
                "https://swapi.dev/api/starships/22/"
            ],
            "created": "2014-12-10T16:42:45.066000Z",
            "edited": "2014-12-20T21:17:50.332000Z",
            "url": "https://swapi.dev/api/people/13/"
        }
    },
    "https://swapi.dev/api/planets/2/": {
        "status_code": 200,
        "json": {
            "name": "Alderaan",
            "rotation_period": "24",
            "orbital_period": "364",
            "diameter": "12500",
            "climate": "temperate",
            "gravity": "1 standard",
            "terrain": "grasslands, mountains",
            "surface_water": "40",
            "population": "2000000000",
            "residents": [
                "https://swapi.dev/api/people/5/",
                "https://swapi.dev/api/people/68/",
                "https://swapi.dev/api/people/81/"
            ],
            "films": [
                "https://swapi.dev/api/films/1/",
                "https://swapi.dev/api/films/6/"
            ],
            "created": "2014-12-10T11:35:48.479000Z",
            "edited": "2014-12-20T20:58:18.420000Z",
            "url": "https://swapi.dev/api/planets/2/"
        }
    },
    "https://swapi.dev/api/starships/10/": {
        "status_code": 200,
        "json": {
            "name": "Millennium Falcon",
            "model": "YT-1300 light freighter",
            "manufacturer": "Corellian Engineering Corporation",
            "cost_in_credits": "100000",
            "length": "34.37",
            "max_atmosphering_speed": "1050",
            "crew": "4",
            "passengers": "6",
            "cargo_capacity": "100000",
            "consumables": "2 months",
            "hyperdrive_rating": "0.5",
            "MGLT": "75",
            "starship_class": "Light freighter",
            "pilots": [
                "https://swapi.dev/api/people/13/",
                "https://swapi.dev/api/people/14/",
                "https://swapi.dev/api/people/25/",
                "https://swapi.dev/api/people/31/"
            ],
            "films": [
                "https://swapi.dev/api/films/1/",
                "https://swapi.dev/api/films/2/",
                "https://swapi.dev/api/films/3/"
            ],
            "created": "2014-12-10T16:59:45.094000Z",
            "edited": "2014-12-20T21:23:49.880000Z",
            "url": "https://swapi.dev/api/starships/10/"
        }
    },
    "https://swapi.dev/api/people/9999/": {
        "status_code": 404,
        "json": {
            "detail": "Not found"
        }
    }
}

class FallbackResponse:
    def __init__(self, url, status_code, payload, source):
        self.url = url
        self.status_code = status_code
        self._payload = payload
        self.source = source
        self.text = json.dumps(payload, ensure_ascii=False)

    def json(self):
        return self._payload

def get_swapi(url, timeout=20):
    """Tenta buscar online. Se a internet falhar, usa dados locais para a aula."""
    try:
        resp = requests.get(url, timeout=timeout)
        try:
            payload = resp.json()
        except Exception:
            payload = {"raw_text": resp.text}
        return FallbackResponse(url, resp.status_code, payload, "online")
    except Exception:
        fixture = FIXTURES.get(url)
        if fixture is None:
            raise
        return FallbackResponse(url, fixture["status_code"], fixture["json"], "fallback_local")

def mostrar_resposta(url):
    response = get_swapi(url)
    print("Fonte da resposta:", response.source)
    print("Status code:", response.status_code)
    print(json.dumps(response.json(), indent=4, ensure_ascii=False))
    return response

## Parte 1 — Explorando a API

Consulte os endpoints abaixo:

- `https://swapi.dev/api/people/10/`
- `https://swapi.dev/api/people/13/`
- `https://swapi.dev/api/planets/2/`
- `https://swapi.dev/api/starships/10/`

Complete a célula abaixo.

In [24]:
urls = {
    "personagem_10": "https://swapi.dev/api/people/69/",
    "personagem_13": "https://swapi.dev/api/people/100/",
    "planeta_2": "https://swapi.dev/api/planets/10/",
    "nave_10": "https://swapi.dev/api/starships/5/"
}

respostas = {}
for nome, url in urls.items():
    print("\n" + "="*70)
    print("Endpoint:", nome)
    respostas[nome] = mostrar_resposta(url)

# TODO: percorra a lista de URLs e mostre o status code e o JSON de cada resposta


Endpoint: personagem_10
Fonte da resposta: online
Status code: 200
{
    "name": "Jango Fett",
    "height": "183",
    "mass": "79",
    "hair_color": "black",
    "skin_color": "tan",
    "eye_color": "brown",
    "birth_year": "66BBY",
    "gender": "male",
    "homeworld": "https://swapi.dev/api/planets/53/",
    "films": [
        "https://swapi.dev/api/films/5/"
    ],
    "species": [],
    "vehicles": [],
    "starships": [],
    "created": "2014-12-20T16:54:41.620000Z",
    "edited": "2014-12-20T21:17:50.465000Z",
    "url": "https://swapi.dev/api/people/69/"
}

Endpoint: personagem_13
Fonte da resposta: online
Status code: 404
{
    "detail": "Not found"
}

Endpoint: planeta_2
Fonte da resposta: online
Status code: 200
{
    "name": "Kamino",
    "rotation_period": "27",
    "orbital_period": "463",
    "diameter": "19720",
    "climate": "temperate",
    "gravity": "1 standard",
    "terrain": "ocean",
    "surface_water": "100",
    "population": "1000000000",
    "residen

1. Endpoint: personagem_10
Fonte da resposta: online
Status code: 200

2. Endpoint: personagem_13
Fonte da resposta: online
Status code: 404

3. Endpoint: planeta_2
Fonte da resposta: online
Status code: 200

4. Endpoint: nave_10
Fonte da resposta: online
Status code: 200

## Parte 2 — Análise QA da resposta

Responda para cada endpoint:

1. Qual foi o status code?
2. O JSON parece completo?
3. Quais campos apareceram?
4. Qual campo você considera mais importante para validação?

Escreva suas respostas na célula abaixo.

#Parte 2 - Respostas:

1. Qual foi o status code?

R: O status code foi 200 (sucesso)

2. O JSON parece completo?

R: Sim, pois a quantidade de campos de todos são maiores do que 8. Sendo assim, parece completo

3. Quais campos apareceram?

R: Apareceram: "name", "height", "mass", "hair_color", "skin_color", "eye_color", "birth_year", "gender", "homeworld", "films", "species", "vehicles", "starships", "created", "edited", "url". Alguns com mais, outros com menos campos.

4. Qual campo você considera mais importante para validação?

R: Eu acho que o campo "name" é importante para uma busca específica de um herói, "created" para saber se o herói já está criado, "url" do herói, etc.

In [25]:
analise = {}
for nome, response in respostas.items():
    dados = response.json()
    analise[nome] = {
        "status_code": response.status_code,
        "campos": list(dados.keys()),
        "quantidade_campos": len(dados.keys()),
        "name_existe": "name" in dados,
        "json_parece_completo": len(dados.keys()) >= 8
    }

print(json.dumps(analise, indent=4, ensure_ascii=False))

{
    "personagem_10": {
        "status_code": 200,
        "campos": [
            "name",
            "height",
            "mass",
            "hair_color",
            "skin_color",
            "eye_color",
            "birth_year",
            "gender",
            "homeworld",
            "films",
            "species",
            "vehicles",
            "starships",
            "created",
            "edited",
            "url"
        ],
        "quantidade_campos": 16,
        "name_existe": true,
        "json_parece_completo": true
    },
    "personagem_13": {
        "status_code": 404,
        "campos": [
            "detail"
        ],
        "quantidade_campos": 1,
        "name_existe": false,
        "json_parece_completo": false
    },
    "planeta_2": {
        "status_code": 200,
        "campos": [
            "name",
            "rotation_period",
            "orbital_period",
            "diameter",
            "climate",
            "gravity",
            "t

## Parte 3 — Pensando como QA

Escolha **um** endpoint e proponha **5 validações** que deveriam existir.

Exemplos de ideias:
- status code deve ser 200
- o campo `name` deve existir
- o nome não pode estar vazio
- um campo numérico deve conter número
- uma resposta inválida deve retornar 404

In [26]:
response_personagem = respostas["personagem_10"]
dados_personagem = response_personagem.json()

validacoes_propostas = [
    "status code deve ser 200",
    "campo 'name' deve ser obrigatório",
    "campo 'name' não pode ser um número",
    "campo 'birth_year' não pode ser uma string",
    "campo 'cost_in_credits' não pode ser maior que 999"
]

for item in validacoes_propostas:
    print("-", item)

- status code deve ser 200
- campo 'name' deve ser obrigatório
- campo 'name' não pode ser um número
- campo 'birth_year' não pode ser uma string
- campo 'cost_in_credits' não pode ser maior que 999


## Parte 4 — Testes automatizados com assert

Implemente pelo menos **3 testes automatizados** usando `assert`.

Você pode usar um dos endpoints da atividade.

In [27]:
# Testes do personagem 10
resp_10 = respostas["personagem_10"]
dados_10 = resp_10.json()

assert resp_10.status_code == 200, "Erro: people/10 não retornou 200"
assert "name" in dados_10, "Erro: campo 'name' ausente em people/10"
assert dados_10["name"] != "", "Erro: nome vazio em people/10"

# Testes do planeta 2
resp_planeta = respostas["planeta_2"]
dados_planeta = resp_planeta.json()

assert resp_planeta.status_code == 200, "Erro: planets/2 não retornou 200"
assert "population" in dados_planeta, "Erro: population ausente em planets/2"
assert dados_planeta["population"] != "", "Erro: population vazia em planets/2"

# Testes da nave 10
resp_nave = respostas["nave_10"]
dados_nave = resp_nave.json()

assert resp_nave.status_code == 200, "Erro: starships/10 não retornou 200"
assert "manufacturer" in dados_nave, "Erro: manufacturer ausente em starships/10"
assert dados_nave["manufacturer"] != "", "Erro: manufacturer vazio em starships/10"

print("Todos os testes passaram!")

Todos os testes passaram!


## Bônus — Teste negativo

Agora teste um endpoint inválido:

`https://swapi.dev/api/people/9999/`

O que você espera receber?

In [28]:
# Faça aqui um teste negativo com o endpoint inválido.
# Sua meta é validar que o retorno deve ser 404.

url_invalida = {
    "personagem_10": "https://swapi.dev/api/people/9999/",
}

resposta = {}
for nome, url in url_invalida.items():
    print("\n" + "="*70)
    print("Endpoint:", nome)
    resposta[nome] = mostrar_resposta(url)


Endpoint: personagem_10
Fonte da resposta: online
Status code: 404
{
    "detail": "Not found"
}


R: O status code 404 foi retornado 404 ao testar com esse endpoint

## Entrega

Entregue:

- o notebook preenchido
- sua análise QA
- suas 5 validações
- seus testes com `assert`

Postar em seu REPO no GITHUB